# 04 - Build Senate Money Snapshot

This notebook builds the final MVP snapshot from the top Democratic and Republican candidates per state.
It creates a single clean CSV output for the Senate money snapshot.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

In [ ]:
wide_path = ROOT / "outputs" / "senate_two_party_race_universe_2026.csv"
df_wide = pd.read_csv(wide_path, dtype=str)
print("Loaded two-party race universe shape:", df_wide.shape)
df_wide.head()

In [ ]:
df_top = pd.read_csv(ROOT / "data" / "processed" / "senate_top_dem_rep_candidates_2026.csv", dtype=str)
df_selected = df_top[df_top["selected_candidate"] == True].copy()

def safe_value(df, state, column):
    try:
        return df.loc[state, column]
    except Exception:
        return None

states = sorted(set(df_selected["state"]))
rows = []
for state in states:
    rows.append({
        "state": state,
        "dem_candidate_name": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "selected_candidate_name"),
        "dem_fec_candidate_id": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "fec_candidate_id"),
        "dem_committee_id": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "committee_id"),
        "dem_total_receipts": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "total_receipts"),
        "dem_total_disbursements": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "total_disbursements"),
        "dem_cash_on_hand": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "cash_on_hand_end_period") or safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "cash_on_hand"),
        "dem_debts_owed_by_committee": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "debts_owed_by_committee"),
        "dem_coverage_end_date": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "coverage_end_date"),
        "dem_selection_method": safe_value(df_selected[df_selected["party_normalized"] == "DEM"], state, "selection_method"),
        "rep_candidate_name": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "selected_candidate_name"),
        "rep_fec_candidate_id": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "fec_candidate_id"),
        "rep_committee_id": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "committee_id"),
        "rep_total_receipts": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "total_receipts"),
        "rep_total_disbursements": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "total_disbursements"),
        "rep_cash_on_hand": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "cash_on_hand_end_period") or safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "cash_on_hand"),
        "rep_debts_owed_by_committee": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "debts_owed_by_committee"),
        "rep_coverage_end_date": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "coverage_end_date"),
        "rep_selection_method": safe_value(df_selected[df_selected["party_normalized"] == "REP"], state, "selection_method"),
    })

_df_snapshot = pd.DataFrame(rows)
_df_snapshot["direct_money_data_pulled_at"] = pd.Timestamp.now().isoformat()
_df_snapshot["notes"] = "Final Senate money snapshot based on current top candidate selection."
_df_snapshot["dem_minus_rep_total_receipts"] = pd.to_numeric(_df_snapshot["dem_total_receipts"], errors="coerce").fillna(0) - pd.to_numeric(_df_snapshot["rep_total_receipts"], errors="coerce").fillna(0)
_df_snapshot["dem_minus_rep_cash_on_hand"] = pd.to_numeric(_df_snapshot["dem_cash_on_hand"], errors="coerce").fillna(0) - pd.to_numeric(_df_snapshot["rep_cash_on_hand"], errors="coerce").fillna(0)
_df_snapshot.head()


In [ ]:
final_path = ROOT / "outputs" / "senate_money_snapshot_2026.csv"
df_snapshot.to_csv(final_path, index=False)
print("Saved final snapshot to:", final_path)